# Video → Multimodal Semantic 3D Scene Search (VGGT-Omega + SAM 3)

This notebook runs the full submission pipeline: reconstruction, pixel-aligned semantics, open-vocabulary 3D querying, and object-level scene memory.


## 1. Clone the repo and install Python dependencies
We use `requirements-colab.txt` because Colab already provides matched Torch / CUDA wheels — reinstalling Torch wastes ~3 minutes and sometimes breaks the runtime.

In [ ]:
!git clone https://github.com/ayushmaankaria/Video-to-3D-Reconstruction.git
%cd Video-to-3D-Reconstruction

!pip install -q -r requirements-colab.txt

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Authenticate with Hugging Face
Both VGGT-Omega and SAM 3 are gated. Before running this cell:

1. Request access at https://huggingface.co/facebook/vggt-omega and https://huggingface.co/facebook/sam3 .
2. Create a *Read* token at https://huggingface.co/settings/tokens .
3. In Colab's left sidebar, open the 🔑 **Secrets** panel, add a secret named `HF_TOKEN`, paste your token, and enable notebook access.

The cell below reads that secret and logs in non-interactively.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
assert token, "Add an HF_TOKEN secret in the Colab Secrets panel and re-run."
login(token=token.strip())
print("Logged in to Hugging Face.")

## 3a. (Option A) Use a video from Google Drive
Mount Drive and point `VIDEO_PATH` at your phone video. Skip this cell and use 3b instead if you'd rather upload directly.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Change this to the exact location of your video in Drive.
VIDEO_PATH = "/content/drive/MyDrive/desk_video.mp4"
!test -f "$VIDEO_PATH" && echo "Using $VIDEO_PATH" || echo "Video not found. Update VIDEO_PATH or use the upload fallback cell."

## 3b. (Option B) Upload a video directly
Pick a small video (under ~100 MB is fine). The selected file's path becomes `VIDEO_PATH`.

In [ ]:
from google.colab import files
uploaded = files.upload()
video_name = next(iter(uploaded.keys()))
VIDEO_PATH = f"/content/Video-to-3D-Reconstruction/{video_name}"
print("Using", VIDEO_PATH)

## 4. Download the VGGT-Omega checkpoint
Grabs the 512-resolution 1B checkpoint into `checkpoints/`. If you'd rather use the 256 + text-alignment variant, swap the repo id and pass `--image-resolution 256 --enable-alignment` in the next cell.

In [ ]:
import os
from huggingface_hub import hf_hub_download

CORRECT_REPO_ID = "facebook/VGGT-Omega"
CHECKPOINT_FILE_NAME = "vggt_omega_1b_512.pt"
LOCAL_CKPT_DIR = "checkpoints/vggt-omega-1b-512"
FINAL_MODEL_PATH = os.path.join(LOCAL_CKPT_DIR, "model.pt")

# Ensure the local directory exists
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

# Download the specific checkpoint file
try:
    downloaded_file_path = hf_hub_download(
        repo_id=CORRECT_REPO_ID,
        filename=CHECKPOINT_FILE_NAME,
        local_dir=LOCAL_CKPT_DIR,
        # local_dir_use_symlinks=False # This argument is deprecated and ignored
    )
    print(f"File downloaded to: {downloaded_file_path}")

    # Rename the downloaded file to 'model.pt' as expected by the next cell
    if os.path.exists(downloaded_file_path):
        os.rename(downloaded_file_path, FINAL_MODEL_PATH)
        print(f"Checkpoint renamed to: {FINAL_MODEL_PATH}")
    else:
        print(f"Error: Downloaded file not found at {downloaded_file_path}")

except Exception as e:
    print(f"Error downloading checkpoint: {e}")
    downloaded_file_path = None # Indicate failure

print("Checkpoint directory:", LOCAL_CKPT_DIR)

## 5. Run the reconstruction pipeline
This single CLI call does:
1. Sample frames from the video (`--max-frames`, hybrid sharp+uniform mode).
2. Run VGGT-Omega for dense depth + camera intrinsics/extrinsics.
3. Run SAM 3 once per frame for each text concept in `--concepts` and stamp out a label map.
4. Fuse everything into a colored, semantically-labeled point cloud and write `runs/desk/exports/`.

Tweak `--concepts` to match what's actually in your scene — extra concepts cost roughly +0.3 s/frame each.

In [ ]:
!python -m spatial_recon.cli run \
  --video "$VIDEO_PATH" \
  --out runs/desk \
  --checkpoint checkpoints/vggt-omega-1b-512/model.pt \
  --image-resolution 512 \
  --max-frames 48 \
  --fps 2.0 \
  --concepts "desk,chair,monitor,keyboard,mouse,cup,wall,floor,lamp,cable,guitar,toothbrush,medicine,watch" \
  --conf-percentile 30 \
  --sample-stride 2 \
  --voxel-size 0.01

## 6. Preview the interactive viewer inline
`viewer.html` is a self-contained Plotly scene — colored points, semantic toggle, and the camera trajectory. Rendering inside Colab works for clouds up to ~500k points; for bigger ones, download and open locally.

In [ ]:
from IPython.display import HTML, display
display(HTML(filename="runs/desk/exports/viewer.html"))

## 7. Open-vocabulary 3D query (SAM 3 backed)
Re-runs SAM 3 with your text prompt over the original frames, then highlights the top-K% of fused points whose source pixels fall inside the highest-scoring SAM 3 masks. Output is `runs/desk/exports/query_<text>.ply`.

In [ ]:
!python -m spatial_recon.cli query \
  --run runs/desk \
  --text "chair" \
  --topk-percent 8

## 8. Build object-level scene memory
This step turns the semantic point cloud into object-level scene memory. It clusters points by semantic class, estimates each object's centroid / extent / volume / confidence, adds simple affordances, and writes visual verification files.


In [ ]:
!python -m spatial_recon.memory \
  --run runs/desk \
  --eps 0.15 \
  --min-samples 20

## 9. Preview the object-instance memory viewer
`memory_instances.html` colors each clustered object separately and labels centroids, so the printed memory queries can be checked against the actual 3D geometry. Toggle traces in the legend to inspect individual objects like `desk_01`, `chair_01`, or `keyboard_01`.


In [ ]:
from IPython.display import HTML, display
display(HTML(filename="runs/desk/exports/memory_instances.html"))

## 10. Query the scene memory
These queries are answered from `scene_memory.json`, not directly from an LLM. For example, `desk_01` is returned for placeable surfaces because the object was clustered as a desk and assigned the `placeable` affordance. Nearness is computed from 3D centroid distance.


In [ ]:
!python -m spatial_recon.query_scene --run runs/desk

## 11. Zip and download all outputs
Bundles frames, VGGT-Omega predictions, SAM 3 label maps, fused PLY/GLB outputs, the viewer HTML, open-vocabulary query PLYs, scene memory files, and `REPORT.md`.


In [ ]:
from google.colab import files
!zip -r desk_reconstruction_outputs.zip runs/desk
files.download("desk_reconstruction_outputs.zip")